# Module 4: Panel Model Diagnostics

## Learning Objectives

By completing this notebook, you will:
1. Test for heteroskedasticity in panel data models
2. Detect and correct for serial correlation in errors
3. Identify cross-sectional dependence across entities
4. Implement robust standard errors and corrections
5. Assess overall model adequacy and specification

## Prerequisites

- Module 2 and 3 completed (FE and RE)
- Module 4.1 (Model Selection) completed
- Understanding of OLS assumptions

## Estimated Time

75-90 minutes

---

## Setup

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, het_breuschpagan
from linearmodels.panel import PanelOLS, RandomEffects
from linearmodels.iv.model import compare

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seed
np.random.seed(42)

print("✅ Setup complete")

In [ ]:
section_divider("1. Panel Data Assumptions Recap")

In [ ]:
learning_objectives(["Test for heteroskedasticity in panel data models", "Detect and correct for serial correlation in errors", "Identify cross-sectional dependence across entities", "Implement robust standard errors and corrections", "Assess overall model adequacy and specification"])

In [ ]:
apply_course_theme()
apply_plot_theme()

## 1. Panel Data Assumptions Recap

### Fixed Effects Assumptions

1. **Strict exogeneity (conditional):** $E[\epsilon_{it} | X_{i1}, ..., X_{iT}, \alpha_i] = 0$

2. **No perfect multicollinearity**

3. **Homoskedasticity:** $Var(\epsilon_{it} | X_{it}) = \sigma^2$

4. **No serial correlation:** $Cov(\epsilon_{it}, \epsilon_{is}) = 0$ for $t \neq s$

5. **No cross-sectional dependence:** $Cov(\epsilon_{it}, \epsilon_{jt}) = 0$ for $i \neq j$

### What Can Go Wrong?

- **Heteroskedasticity:** Error variance differs across entities or time → Standard errors biased
- **Serial correlation:** Errors correlated within entity over time → Standard errors biased
- **Cross-sectional dependence:** Errors correlated across entities → Standard errors biased

**Consequence:** Coefficients remain consistent, but inference (t-tests, confidence intervals) is invalid.

In [ ]:
section_divider("2. Generate Data with Violations")

## 2. Generate Data with Violations

Create datasets with different diagnostic issues.

In [ ]:
def generate_panel_with_heteroskedasticity(n=100, t=10, beta=2.0):
    """
    Generate panel data with heteroskedasticity.
    Error variance increases with entity size.
    """
    alpha_i = np.random.randn(n) * 10 + 50
    entity_size = np.random.exponential(scale=2.0, size=n) + 1  # Varies across entities
    
    data = []
    for i in range(n):
        for t_idx in range(t):
            x = 10 + np.random.randn() * 2
            
            # Error variance proportional to entity size (heteroskedasticity)
            error_std = 5 * np.sqrt(entity_size[i])
            epsilon = np.random.randn() * error_std
            
            y = alpha_i[i] + beta * x + epsilon
            
            data.append({
                'entity': i + 1,
                'time': t_idx + 1,
                'y': y,
                'x': x,
                'entity_size': entity_size[i]
            })
    
    return pd.DataFrame(data)

def generate_panel_with_serial_correlation(n=100, t=10, beta=2.0, rho=0.5):
    """
    Generate panel data with AR(1) serial correlation in errors.
    epsilon_it = rho * epsilon_{i,t-1} + nu_it
    """
    alpha_i = np.random.randn(n) * 10 + 50
    
    data = []
    for i in range(n):
        epsilon_prev = 0
        
        for t_idx in range(t):
            x = 10 + np.random.randn() * 2
            
            # AR(1) error
            nu = np.random.randn() * 5
            epsilon = rho * epsilon_prev + nu
            epsilon_prev = epsilon
            
            y = alpha_i[i] + beta * x + epsilon
            
            data.append({
                'entity': i + 1,
                'time': t_idx + 1,
                'y': y,
                'x': x
            })
    
    return pd.DataFrame(data)

# Generate datasets
df_hetero = generate_panel_with_heteroskedasticity()
df_serial = generate_panel_with_serial_correlation(rho=0.6)

print("Generated datasets:")
print(f"  1. Heteroskedastic errors: {len(df_hetero)} obs")
print(f"  2. Serially correlated errors (ρ=0.6): {len(df_serial)} obs")

In [ ]:
section_divider("3. Heteroskedasticity Tests")

## 3. Heteroskedasticity Tests

### Modified Wald Test for Groupwise Heteroskedasticity

**H0:** $\sigma^2_1 = \sigma^2_2 = ... = \sigma^2_n$ (homoskedastic)

**Ha:** At least one $\sigma^2_i$ differs

In [ ]:
def modified_wald_test(df, entity_col, residuals):
    """
    Modified Wald test for groupwise heteroskedasticity.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    residuals : array-like
        Residuals from FE estimation
    
    Returns
    -------
    dict
        Test results
    """
    df_test = df.copy()
    df_test['residual'] = residuals
    
    # Compute entity-specific variances
    entity_vars = df_test.groupby(entity_col)['residual'].var()
    n_entities = len(entity_vars)
    
    # Wald statistic
    # Under H0: sum of log(sigma_i^2) ~ chi2
    overall_var = df_test['residual'].var()
    wald_stat = n_entities * (np.log(overall_var) - np.mean(np.log(entity_vars)))
    
    # Alternative formulation
    wald_stat_alt = np.sum((entity_vars / overall_var - 1)**2)
    
    # Degrees of freedom
    df_wald = n_entities - 1
    
    # P-value
    p_value = 1 - stats.chi2.cdf(wald_stat_alt, df_wald)
    
    return {
        'wald_statistic': wald_stat_alt,
        'df': df_wald,
        'p_value': p_value,
        'entity_variances': entity_vars
    }

# Estimate FE on heteroskedastic data
df_hetero_panel = df_hetero.set_index(['entity', 'time'])
model_fe_hetero = PanelOLS(
    dependent=df_hetero_panel['y'],
    exog=df_hetero_panel[['x']],
    entity_effects=True
)
results_fe_hetero = model_fe_hetero.fit()

# Test for heteroskedasticity
wald_test = modified_wald_test(df_hetero, 'entity', results_fe_hetero.resids)

print("\nModified Wald Test for Groupwise Heteroskedasticity")
print("="*70)
print(f"H0: Homoskedastic errors (equal variance across entities)\n")
print(f"Wald statistic: {wald_test['wald_statistic']:.4f}")
print(f"Degrees of freedom: {wald_test['df']}")
print(f"P-value: {wald_test['p_value']:.4e}")

if wald_test['p_value'] < 0.05:
    print("\n✅ Reject H0: Heteroskedasticity detected")
    print("   → Use robust standard errors")
else:
    print("\n❌ Fail to reject H0: No evidence of heteroskedasticity")

### Visualize Heteroskedasticity

In [ ]:
# Plot residuals vs entity size
df_hetero_with_resid = df_hetero.copy()
df_hetero_with_resid['residual'] = results_fe_hetero.resids.values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot
axes[0].scatter(df_hetero_with_resid['entity_size'], 
                np.abs(df_hetero_with_resid['residual']), 
                alpha=0.3, s=20)
axes[0].set_xlabel('Entity Size', fontsize=11)
axes[0].set_ylabel('Absolute Residual', fontsize=11)
axes[0].set_title('Heteroskedasticity Pattern', fontsize=12)
axes[0].grid(True, alpha=0.3)

# Box plot by entity size quartile
df_hetero_with_resid['size_quartile'] = pd.qcut(df_hetero_with_resid['entity_size'], 
                                                  q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
df_hetero_with_resid.boxplot(column='residual', by='size_quartile', ax=axes[1])
axes[1].set_xlabel('Entity Size Quartile', fontsize=11)
axes[1].set_ylabel('Residual', fontsize=11)
axes[1].set_title('Residuals by Entity Size', fontsize=12)
axes[1].get_figure().suptitle('')  # Remove auto title

plt.tight_layout()
plt.show()

print("Variance increases with entity size → Heteroskedasticity")

In [ ]:
section_divider("4. Serial Correlation Tests")

## 4. Serial Correlation Tests

### Wooldridge Test for Serial Correlation

Tests for AR(1) serial correlation in panel data.

**H0:** No first-order serial correlation

In [ ]:
def wooldridge_test(df, entity_col, time_col, residuals):
    """
    Wooldridge test for serial correlation in panel data.
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    residuals : array-like
    
    Returns
    -------
    dict
        Test results
    """
    df_test = df.copy()
    df_test['residual'] = residuals
    df_test = df_test.sort_values([entity_col, time_col])
    
    # Compute lagged residuals within entity
    df_test['residual_lag'] = df_test.groupby(entity_col)['residual'].shift(1)
    
    # Drop first period (no lag)
    df_test_clean = df_test.dropna(subset=['residual_lag'])
    
    # Regress residual on lagged residual
    X = sm.add_constant(df_test_clean['residual_lag'])
    y = df_test_clean['residual']
    
    model = sm.OLS(y, X)
    results = model.fit()
    
    # Test statistic: t-test for rho=0
    rho = results.params['residual_lag']
    t_stat = results.tvalues['residual_lag']
    p_value = results.pvalues['residual_lag']
    
    # F-test alternative
    f_stat = results.fvalue
    f_pvalue = results.f_pvalue
    
    return {
        'rho': rho,
        't_statistic': t_stat,
        'p_value': p_value,
        'f_statistic': f_stat,
        'f_pvalue': f_pvalue
    }

# Estimate FE on serially correlated data
df_serial_panel = df_serial.set_index(['entity', 'time'])
model_fe_serial = PanelOLS(
    dependent=df_serial_panel['y'],
    exog=df_serial_panel[['x']],
    entity_effects=True
)
results_fe_serial = model_fe_serial.fit()

# Test for serial correlation
wooldridge_result = wooldridge_test(df_serial, 'entity', 'time', results_fe_serial.resids)

print("\nWooldridge Test for Serial Correlation")
print("="*70)
print(f"H0: No first-order serial correlation (ρ = 0)\n")
print(f"Estimated ρ: {wooldridge_result['rho']:.4f}")
print(f"t-statistic: {wooldridge_result['t_statistic']:.4f}")
print(f"P-value: {wooldridge_result['p_value']:.4e}")

if wooldridge_result['p_value'] < 0.05:
    print("\n✅ Reject H0: Serial correlation detected")
    print(f"   → Autocorrelation coefficient: {wooldridge_result['rho']:.2f}")
    print("   → Use cluster-robust standard errors or correct for serial correlation")
else:
    print("\n❌ Fail to reject H0: No evidence of serial correlation")

### Visualize Serial Correlation

In [ ]:
# ACF plot for residuals
df_serial_with_resid = df_serial.copy()
df_serial_with_resid['residual'] = results_fe_serial.resids.values
df_serial_with_resid = df_serial_with_resid.sort_values(['entity', 'time'])

# Compute residual lag
df_serial_with_resid['residual_lag'] = df_serial_with_resid.groupby('entity')['residual'].shift(1)
df_plot = df_serial_with_resid.dropna(subset=['residual_lag'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: e_t vs e_{t-1}
axes[0].scatter(df_plot['residual_lag'], df_plot['residual'], alpha=0.3, s=20)

# Add regression line
z = np.polyfit(df_plot['residual_lag'], df_plot['residual'], 1)
p = np.poly1d(z)
x_line = np.linspace(df_plot['residual_lag'].min(), df_plot['residual_lag'].max(), 100)
axes[0].plot(x_line, p(x_line), 'r--', linewidth=2, label=f'ρ = {z[0]:.2f}')

axes[0].set_xlabel('Residual (t-1)', fontsize=11)
axes[0].set_ylabel('Residual (t)', fontsize=11)
axes[0].set_title('Serial Correlation in Residuals', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Time series for sample entity
sample_entity = df_serial_with_resid['entity'].iloc[0]
entity_data = df_serial_with_resid[df_serial_with_resid['entity'] == sample_entity].sort_values('time')

axes[1].plot(entity_data['time'], entity_data['residual'], marker='o', linewidth=2)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1)
axes[1].set_xlabel('Time', fontsize=11)
axes[1].set_ylabel('Residual', fontsize=11)
axes[1].set_title(f'Residual Time Series (Entity {sample_entity})', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Positive slope indicates positive serial correlation")

In [ ]:
section_divider("5. Robust Standard Errors")

## 5. Robust Standard Errors

When diagnostics detect violations, use robust standard errors.

In [ ]:
def compare_standard_errors(df_panel, y_col='y', x_cols=['x']):
    """
    Compare different standard error calculations.
    """
    results_dict = {}
    
    # Standard (non-robust)
    model = PanelOLS(
        dependent=df_panel[y_col],
        exog=df_panel[x_cols],
        entity_effects=True
    )
    results_dict['Standard'] = model.fit()
    
    # Heteroskedasticity-robust (White)
    results_dict['Robust'] = model.fit(cov_type='robust')
    
    # Clustered by entity (corrects for heteroskedasticity and serial correlation)
    results_dict['Clustered'] = model.fit(cov_type='clustered', cluster_entity=True)
    
    return results_dict

# Compare on heteroskedastic data
se_comparison_hetero = compare_standard_errors(df_hetero_panel)

print("\nStandard Error Comparison (Heteroskedastic Data)")
print("="*70)
print(f"{'Method':<20} {'Coef':>12} {'Std Error':>12} {'t-stat':>10} {'P-value':>10}")
print("="*70)

for method, results in se_comparison_hetero.items():
    coef = results.params['x']
    se = results.std_errors['x']
    t_stat = results.tstats['x']
    p_val = results.pvalues['x']
    
    print(f"{method:<20} {coef:>12.4f} {se:>12.4f} {t_stat:>10.3f} {p_val:>10.4f}")

print("\nNote: Robust and clustered SEs are larger (correct for heteroskedasticity)")

### Compare on Serially Correlated Data

In [ ]:
# Compare on serially correlated data
se_comparison_serial = compare_standard_errors(df_serial_panel)

print("\nStandard Error Comparison (Serially Correlated Data)")
print("="*70)
print(f"{'Method':<20} {'Coef':>12} {'Std Error':>12} {'t-stat':>10} {'P-value':>10}")
print("="*70)

for method, results in se_comparison_serial.items():
    coef = results.params['x']
    se = results.std_errors['x']
    t_stat = results.tstats['x']
    p_val = results.pvalues['x']
    
    print(f"{method:<20} {coef:>12.4f} {se:>12.4f} {t_stat:>10.3f} {p_val:>10.4f}")

print("\nNote: Clustered SEs correct for both heteroskedasticity and serial correlation")

## 6. Cross-Sectional Dependence

Errors may be correlated across entities (common shocks).

In [ ]:
def pesaran_cd_test(df, entity_col, time_col, residuals):
    """
    Pesaran CD test for cross-sectional dependence.
    
    H0: No cross-sectional dependence
    
    Parameters
    ----------
    df : pd.DataFrame
    entity_col : str
    time_col : str
    residuals : array-like
    
    Returns
    -------
    dict
        Test results
    """
    df_test = df.copy()
    df_test['residual'] = residuals
    
    # Reshape to wide format (entities × time)
    resid_wide = df_test.pivot(index=time_col, columns=entity_col, values='residual')
    
    # Compute correlation matrix
    corr_matrix = resid_wide.corr()
    
    # Extract upper triangle (exclude diagonal)
    n_entities = corr_matrix.shape[0]
    upper_tri_idx = np.triu_indices(n_entities, k=1)
    correlations = corr_matrix.values[upper_tri_idx]
    
    # CD statistic
    n = n_entities
    t = len(resid_wide)
    
    cd_stat = np.sqrt(2 * t / (n * (n - 1))) * np.sum(correlations)
    
    # Under H0: CD ~ N(0, 1)
    p_value = 2 * (1 - stats.norm.cdf(np.abs(cd_stat)))
    
    return {
        'cd_statistic': cd_stat,
        'p_value': p_value,
        'avg_correlation': np.mean(np.abs(correlations)),
        'max_correlation': np.max(np.abs(correlations))
    }

# Generate data with cross-sectional dependence
def generate_panel_with_csd(n=50, t=10, beta=2.0):
    """Generate panel with common time shocks."""
    alpha_i = np.random.randn(n) * 10 + 50
    
    # Common time shocks
    time_shocks = np.random.randn(t) * 5
    
    data = []
    for i in range(n):
        for t_idx in range(t):
            x = 10 + np.random.randn() * 2
            
            # Idiosyncratic + common shock
            epsilon = np.random.randn() * 3 + time_shocks[t_idx]
            
            y = alpha_i[i] + beta * x + epsilon
            
            data.append({
                'entity': i + 1,
                'time': t_idx + 1,
                'y': y,
                'x': x
            })
    
    return pd.DataFrame(data)

df_csd = generate_panel_with_csd()
df_csd_panel = df_csd.set_index(['entity', 'time'])

# Estimate and test
model_fe_csd = PanelOLS(
    dependent=df_csd_panel['y'],
    exog=df_csd_panel[['x']],
    entity_effects=True
)
results_fe_csd = model_fe_csd.fit()

cd_test = pesaran_cd_test(df_csd, 'entity', 'time', results_fe_csd.resids)

print("\nPesaran CD Test for Cross-Sectional Dependence")
print("="*70)
print(f"H0: No cross-sectional dependence\n")
print(f"CD statistic: {cd_test['cd_statistic']:.4f}")
print(f"P-value: {cd_test['p_value']:.4f}")
print(f"Average absolute correlation: {cd_test['avg_correlation']:.4f}")
print(f"Maximum absolute correlation: {cd_test['max_correlation']:.4f}")

if cd_test['p_value'] < 0.05:
    print("\n✅ Reject H0: Cross-sectional dependence detected")
    print("   → Consider time fixed effects or Driscoll-Kraay SEs")
else:
    print("\n❌ Fail to reject H0: No evidence of cross-sectional dependence")

## 7. Comprehensive Diagnostic Report

Create a function that runs all diagnostics.

In [ ]:
def comprehensive_diagnostics(df, entity_col, time_col, y_col, x_cols):
    """
    Run comprehensive diagnostic tests on panel data.
    
    Returns
    -------
    dict
        All diagnostic results and recommendations
    """
    # Estimate FE model
    df_panel = df.set_index([entity_col, time_col])
    model = PanelOLS(
        dependent=df_panel[y_col],
        exog=df_panel[x_cols],
        entity_effects=True
    )
    results = model.fit()
    
    diagnostics = {}
    recommendations = []
    
    # 1. Heteroskedasticity test
    try:
        wald = modified_wald_test(df, entity_col, results.resids)
        diagnostics['heteroskedasticity'] = wald
        
        if wald['p_value'] < 0.05:
            recommendations.append("⚠️  Heteroskedasticity detected → Use robust SEs")
    except:
        diagnostics['heteroskedasticity'] = {'error': 'Test failed'}
    
    # 2. Serial correlation test
    try:
        wooldridge = wooldridge_test(df, entity_col, time_col, results.resids)
        diagnostics['serial_correlation'] = wooldridge
        
        if wooldridge['p_value'] < 0.05:
            recommendations.append(f"⚠️  Serial correlation detected (ρ={wooldridge['rho']:.2f}) → Use clustered SEs")
    except:
        diagnostics['serial_correlation'] = {'error': 'Test failed'}
    
    # 3. Cross-sectional dependence test
    try:
        cd = pesaran_cd_test(df, entity_col, time_col, results.resids)
        diagnostics['cross_sectional_dependence'] = cd
        
        if cd['p_value'] < 0.05:
            recommendations.append("⚠️  Cross-sectional dependence → Consider time FE or Driscoll-Kraay SEs")
    except:
        diagnostics['cross_sectional_dependence'] = {'error': 'Test failed'}
    
    # Final recommendation
    if len(recommendations) == 0:
        recommendations.append("✅ No violations detected → Standard SEs are valid")
    elif len(recommendations) >= 2:
        recommendations.append("📌 Recommendation: Use clustered SEs (robust to multiple violations)")
    
    diagnostics['recommendations'] = recommendations
    diagnostics['results'] = results
    
    return diagnostics

# Run on firm dataset
diag_report = comprehensive_diagnostics(df_hetero, 'entity', 'time', 'y', ['x'])

print("\n" + "="*70)
print("COMPREHENSIVE DIAGNOSTIC REPORT")
print("="*70)

print("\n1. Heteroskedasticity Test:")
if 'error' not in diag_report['heteroskedasticity']:
    print(f"   Wald statistic: {diag_report['heteroskedasticity']['wald_statistic']:.2f}")
    print(f"   P-value: {diag_report['heteroskedasticity']['p_value']:.4f}")
    print(f"   Result: {'DETECTED' if diag_report['heteroskedasticity']['p_value'] < 0.05 else 'NOT DETECTED'}")

print("\n2. Serial Correlation Test:")
if 'error' not in diag_report['serial_correlation']:
    print(f"   Estimated ρ: {diag_report['serial_correlation']['rho']:.4f}")
    print(f"   P-value: {diag_report['serial_correlation']['p_value']:.4f}")
    print(f"   Result: {'DETECTED' if diag_report['serial_correlation']['p_value'] < 0.05 else 'NOT DETECTED'}")

print("\n3. Cross-Sectional Dependence Test:")
if 'error' not in diag_report['cross_sectional_dependence']:
    print(f"   CD statistic: {diag_report['cross_sectional_dependence']['cd_statistic']:.4f}")
    print(f"   P-value: {diag_report['cross_sectional_dependence']['p_value']:.4f}")
    print(f"   Result: {'DETECTED' if diag_report['cross_sectional_dependence']['p_value'] < 0.05 else 'NOT DETECTED'}")

print("\n" + "="*70)
print("RECOMMENDATIONS:")
print("="*70)
for rec in diag_report['recommendations']:
    print(f"  {rec}")
print("="*70)

---

## Exercises

### Exercise 1: Implement Breusch-Pagan Test

**Task:** Implement the Breusch-Pagan test for heteroskedasticity in panel data.

In [ ]:
# YOUR CODE HERE

your_bp_test = None  # Replace with implementation

### Exercise 2: Durbin-Watson Statistic

**Task:** Compute Durbin-Watson statistic for each entity and report average.

In [ ]:
# YOUR CODE HERE

dw_stats = None  # Replace with implementation

### Exercise 3: Compare Inference

**Task:** Show how standard error choice affects hypothesis testing conclusions.

In [ ]:
# YOUR CODE HERE

inference_comparison = None  # Replace with implementation

---

## Summary

### Key Takeaways

1. **Diagnostic Tests Essential:** Always test assumptions before trusting inference

2. **Common Violations:**
   - Heteroskedasticity: Variance differs across entities
   - Serial correlation: Errors correlated within entity
   - Cross-sectional dependence: Errors correlated across entities

3. **Robust Solutions:**
   - Heteroskedasticity-robust SEs (White)
   - Cluster-robust SEs (corrects for heteroskedasticity + serial correlation)
   - Driscoll-Kraay SEs (corrects for all three)

4. **Coefficients Still Consistent:** Violations affect inference, not consistency

5. **Default Recommendation:** Always use cluster-robust SEs in panel data

### Standard Error Decision Tree

```
Panel Data Model
├─ Heteroskedasticity only → Robust SEs
├─ Serial correlation only → Clustered SEs
├─ Both hetero + serial → Clustered SEs
└─ + Cross-sectional dep → Driscoll-Kraay SEs or Time FE

Default: Use Clustered SEs (conservative, widely accepted)
```

### What's Next

Module 5:
- Dynamic panel models
- Lagged dependent variables
- GMM estimation
- Advanced topics

---

**Next:** Module 5 - Advanced Panel Methods

In [ ]:
key_takeaways(["Next:"])